In [1]:
pip install gradio

In [2]:
import gradio as gr

In [5]:
pip install langchain_community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.1 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [10]:
pip install langchain

In [11]:
pip install transformers

In [6]:
pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 332.2/332.2 kB 20.0 MB/s eta 0:00:00


In [13]:
import gradio as gr
from transformers import pipeline
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter

In [14]:
qa_pipeline = pipeline("question-answering", model="distilbert-base-cased-distilled-squad")

pdf_context = ""

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

In [15]:
def process_pdf(file_obj):
    """Extracts text from the uploaded PDF file."""
    global pdf_context
    if file_obj is None:
        return "❌ Error: No file uploaded."

    try:
        # Load PDF using the file path provided by Gradio
        loader = PyPDFLoader(file_obj.name)
        pages = loader.load()

        # Split text into manageable chunks
        text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
        chunks = text_splitter.split_documents(pages)

        # Join chunks into one searchable string
        pdf_context = " ".join([c.page_content for c in chunks])

        print(f"Success! Extracted {len(pdf_context)} characters.")
        return "✅ PDF processed successfully! You can now ask questions."
    except Exception as e:
        print(e)

def chat_logic(user_question, history):
    """Handles the Question & Answering logic."""
    global pdf_context

    # Safety check: if no PDF is uploaded
    if not pdf_context or pdf_context.strip() == "":
        history.append((user_question, "⚠️ Please upload and 'Process' a PDF first."))
        return "", history

    try:

        context_slice = pdf_context[:3000]

        result = qa_pipeline(question=user_question, context=context_slice)
        answer = result["answer"]

    except:
        print('Error')

    # Update history for the Gradio Chatbot component
    history.append((user_question, answer))

    # Return "" to clear the input textbox, and the updated history
    return "", history

# --------------------------
# 3. Gradio Interface Construction
# --------------------------

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🤖 AI PDF Assistant")
    gr.Markdown("Upload a PDF and ask questions about its content. Uses DistilBERT (Local CPU).")

    with gr.Row():
        with gr.Column(scale=1):
            file_input = gr.File(label="1. Upload PDF", file_types=[".pdf"])
            process_btn = gr.Button("2. Process PDF", variant="primary")
            status_output = gr.Textbox(label="System Status", interactive=False)

        with gr.Column(scale=2):
            chatbot_display = gr.Chatbot(label="Conversation", height=400)
            question_input = gr.Textbox(label="3. Ask a Question", placeholder="e.g., What is the main topic?")
            submit_btn = gr.Button("Send Question")


    # When Process PDF is clicked:
    process_btn.click(
        fn=process_pdf,
        inputs=[file_input],
        outputs=[status_output]
    )

    # When Send Question is clicked (or Enter is pressed):
    submit_event = submit_btn.click(
        fn=chat_logic,
        inputs=[question_input, chatbot_display],
        outputs=[question_input, chatbot_display]
    )

    question_input.submit(
        fn=chat_logic,
        inputs=[question_input, chatbot_display],
        outputs=[question_input, chatbot_display]
    )

# --------------------------
# 4. Launch
# --------------------------
if __name__ == "__main__":
    demo.launch()

/tmp/ipykernel_4203/1177205251.py:53: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:
/tmp/ipykernel_4203/1177205251.py:64: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot_display = gr.Chatbot(label="Conversation", height=400)
/tmp/ipykernel_4203/1177205251.py:64: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot_display = gr.Chatbot(label="Conversation", height=400)


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://07dc1ee97aeb086e9d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
